# Part 9: CAR Archives

CAR (Content Addressable aRchive) is the file format ATProto uses to
export and import repositories. A CAR file is a sequence of CBOR-encoded
blocks, each identified by its CID. The first block is always the repo root.

**What you'll learn:**
- CAR v1 header structure (version, roots, characteristics)
- Block storage: CID-prefixed CBOR data
- CAR reader: iterating blocks in order
- CAR writer: assembling blocks into a valid CAR

**Prerequisites:** Part 7 (CID), Part 8 (DAG-CBOR)

**Estimated Time:** 15-20 minutes

## Step 1: CAR v1 Header

A CAR v1 file starts with a header block:

```
varint(header-length) + CBOR{
  "version": 1,
  "roots": [CID-of-root-block],
  "characteristics": [0, 0]  // optional
}
```

After the header, each block is:

```
varint(block-length) + CID-bytes + CBOR-data
```

In [ ]:
// CAR header model
@interface CARHeader : NSObject
@property (nonatomic, assign) int version;
@property (nonatomic, strong) NSMutableArray *roots;
- (NSDictionary *)toDictionary;
@end

@implementation CARHeader
- (instancetype)init {
    self = [super init];
    if (self) {
        _version = 1;
        _roots = [NSMutableArray array];
    }
    return self;
}
- (NSDictionary *)toDictionary {
    return @{
        @"version": @(_version),
        @"roots": _roots
    };
}
@end

CARHeader *header = [[CARHeader alloc] init];
[header.roots addObject:@"bafyreiroot123"];
NSLog(@"Header: %@", [header toDictionary]);

## Step 2: CAR Block

Each block in a CAR file is a CID followed by its CBOR data.
We model blocks as a simple key-value pair.

In [ ]:
// CAR block model
@interface CARBlock : NSObject
@property (nonatomic, strong) NSString *cid;
@property (nonatomic, strong) NSData *data;
- (id)initWithCID:(NSString *)cid data:(NSData *)data;
@end

@implementation CARBlock
- (instancetype)initWithCID:(NSString *)cid data:(NSData *)data {
    self = [super init];
    if (self) {
        _cid = cid;
        _data = data;
    }
    return self;
}
@end

## Step 3: CAR Writer

The writer assembles a header and a sequence of blocks into a CAR.
In production, this produces a byte stream. Here we model it as
an ordered list of blocks.

In [ ]:
@interface CARWriter : NSObject
@property (nonatomic, strong) CARHeader *header;
@property (nonatomic, strong) NSMutableArray *blocks;
- (void)addBlock:(id)block;
- (int)blockCount;
- (NSArray *)allCIDs;
@end

@implementation CARWriter
- (instancetype)init {
    self = [super init];
    if (self) {
        _header = [[CARHeader alloc] init];
        _blocks = [NSMutableArray array];
    }
    return self;
}
- (void)addBlock:(CARBlock *)block {
    [_blocks addObject:block];
    // First block's CID becomes the root
    if ([_blocks count] == 1) {
        [_header.roots addObject:[block cid]];
    }
}
- (int)blockCount {
    return [_blocks count];
}
- (NSArray *)allCIDs {
    NSMutableArray *cids = [NSMutableArray array];
    for (int i = 0; i < [_blocks count]; i++) {
        [cids addObject:[[_blocks objectAtIndex:i] cid]];
    }
    return cids;
}
@end

CARWriter *writer = [[CARWriter alloc] init];
[writer addBlock:[[CARBlock alloc] initWithCID:@"bafyreiaaa" data:[NSData dataWithBytes:"root" length:4]]];
[writer addBlock:[[CARBlock alloc] initWithCID:@"bafyreibbb" data:[NSData dataWithBytes:"child" length:5]]];
[writer addBlock:[[CARBlock alloc] initWithCID:@"bafyreiccc" data:[NSData dataWithBytes:"leaf" length:4]]];

NSLog(@"Blocks: %d", [writer blockCount]);
NSLog(@"Roots: %@", [[writer header] roots]);
NSLog(@"All CIDs: %@", [writer allCIDs]);

## Step 4: CAR Reader

The reader iterates blocks in order. The first block is always the root.
In production, `CARReader` reads from a byte stream. Here we iterate
the writer's block list.

In [ ]:
@interface CARReader : NSObject
@property (nonatomic, strong) CARWriter *car;
- (id)initWithCar:(id)car;
- (id)rootBlock;
- (id)blockAtCID:(NSString *)cid;
- (int)verifyIntegrity;
@end

@implementation CARReader
- (instancetype)initWithCar:(CARWriter *)car {
    self = [super init];
    if (self) { _car = car; }
    return self;
}
- (CARBlock *)rootBlock {
    if ([[_car blocks] count] == 0) return nil;
    return [[_car blocks] objectAtIndex:0];
}
- (CARBlock *)blockAtCID:(NSString *)cid {
    for (int i = 0; i < [[_car blocks] count]; i++) {
        CARBlock *block = [[_car blocks] objectAtIndex:i];
        if ([[block cid] isEqualToString:cid]) return block;
    }
    return nil;
}
- (int)verifyIntegrity {
    // Check that all root CIDs exist as blocks
    int missing = 0;
    for (NSString *rootCid in [[_car header] roots]) {
        if ([self blockAtCID:rootCid] == nil) {
            NSLog(@"Missing root block: %@", rootCid);
            missing = missing + 1;
        }
    }
    return missing;
}
@end

CARReader *reader = [[CARReader alloc] initWithCar:writer];
CARBlock *root = [reader rootBlock];
NSLog(@"Root CID: %@", [root cid]);

CARBlock *found = [reader blockAtCID:@"bafyreibbb"];
NSLog(@"Found block: %@", [found cid]);

NSLog(@"Missing roots: %d", [reader verifyIntegrity]);

## Step 5: Garazyk Production Anchor

In the Garazyk codebase, `Repository/CAR.h` and `Repository/CAR.m` implement
full CAR support:

- `CARWriter` — writes header + blocks to a byte buffer
- `CARReader` — reads blocks from a byte buffer
- `CARBlock` — represents a CID + data pair
- `CARHeader` — version, roots, characteristics

The production CAR uses real varint encoding and CBOR serialization.
Our notebook uses simplified models that demonstrate the structure.

## Exercise

Add a `removeBlockAtCID:` method to `CARWriter` that removes a block by CID.
If the removed block was a root, update the header accordingly.

Hint: iterate the blocks array, remove the matching entry, and check if
its CID is in the header's roots array.

In [ ]:
// Exercise: implement removeBlockAtCID:
// Your code here...

## Solution

In [ ]:
// Solution: removeBlockAtCID removes block and updates roots
@interface CARWriter2 : NSObject
@property (nonatomic, strong) CARHeader *header;
@property (nonatomic, strong) NSMutableArray *blocks;
- (void)addBlock:(id)block;
- (BOOL)removeBlockAtCID:(NSString *)cid;
@end

@implementation CARWriter2
- (instancetype)init {
    self = [super init];
    if (self) {
        _header = [[CARHeader alloc] init];
        _blocks = [NSMutableArray array];
    }
    return self;
}
- (void)addBlock:(CARBlock *)block {
    [_blocks addObject:block];
    if ([_blocks count] == 1) {
        [_header.roots addObject:[block cid]];
    }
}
- (BOOL)removeBlockAtCID:(NSString *)cid {
    for (int i = 0; i < [_blocks count]; i++) {
        CARBlock *block = [_blocks objectAtIndex:i];
        if ([[block cid] isEqualToString:cid]) {
            [_blocks removeObjectAtIndex:i];
            // Remove from roots if present
            NSMutableArray *newRoots = [NSMutableArray array];
            for (NSString *rootCid in _header.roots) {
                if (![rootCid isEqualToString:cid]) {
                    [newRoots addObject:rootCid];
                }
            }
            _header.roots = newRoots;
            NSLog(@"Removed block: %@", cid);
            return YES;
        }
    }
    return NO;
}
@end

CARWriter2 *w2 = [[CARWriter2 alloc] init];
[w2 addBlock:[[CARBlock alloc] initWithCID:@"bafyreiroot" data:[NSData dataWithBytes:"r" length:1]]];
[w2 addBlock:[[CARBlock alloc] initWithCID:@"bafyreichild" data:[NSData dataWithBytes:"c" length:1]]];

NSLog(@"Before: %d blocks, %d roots", [[w2 blocks] count], [[[w2 header] roots] count]);
[w2 removeBlockAtCID:@"bafyreiroot"];
NSLog(@"After: %d blocks, %d roots", [[w2 blocks] count], [[[w2 header] roots] count]);

## What to Read Next

You now understand how CAR files package blocks. Next:
- **Part 10: MST Insertion** — how MST nodes are stored as CAR blocks
- **Part 11: MST Proofs & Diff** — how MST diffs produce CAR streams
- **Part 18: Archaeology — Repository** — how production MST/CAR/CBOR code works